# CrewAI Product Launch Crew

**Project:** Multi-Agent Product Launch Brief Generator  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent product launch crew** that collaborates to produce a single, unified launch brief — from strategic planning through market analysis, campaign design, and quality review.

### The Agent Roster

| # | Agent | Role | Deliverable |
|---|-------|------|------------|
| 1 | **Product Manager (PM)** | Strategic Lead | Launch strategy — positioning, audience, success metrics, timeline |
| 2 | **Marketing Strategist** | Campaign Architect | Go-to-market plan — channels, messaging, content calendar, budget |
| 3 | **Market Analyst** | Data & Competitive Intelligence | Market landscape — TAM/SAM/SOM, competitor map, pricing analysis |
| 4 | **QA Reviewer** | Quality & Consistency Auditor | Launch brief audit — gaps, contradictions, risk flags, final sign-off |

### Crew Orchestration — How Four Agents Produce One Document

```
Product Brief (input)
        │
        ▼
┌───────────────────┐
│  Product Manager  │  PLAN — defines the "what" and "why"
└────────┬──────────┘
         │ Launch strategy (positioning, audience, metrics)
         ▼
    ┌────┴────┐
    │ FORK    │  Two agents work with PM's strategy
    ▼         ▼
┌────────┐ ┌──────────┐
│Marketer│ │ Analyst  │  BUILD — the "how" and "where"
└────┬───┘ └────┬─────┘
     │          │
     └────┬─────┘  Both outputs feed into QA
          ▼
┌───────────────────┐
│   QA Reviewer     │  VERIFY — catches gaps, contradictions, risks
└───────────────────┘
          │
          ▼
  Unified Launch Brief
```

### Why Crew Orchestration Matters

Orchestration is  how you control **which agents see which outputs** and in **what order**. Poor orchestration produces:
- **Redundant work** — agents repeating each other's analysis
- **Contradictions** — the marketer targets one audience while the analyst sizes a different one
- **Blind spots** — nobody checks whether the pieces fit together

Good orchestration produces a document that reads like one mind wrote it, even though four agents contributed. This notebook demonstrates three orchestration patterns:

1. **Sequential pipeline** — each agent builds on all prior outputs (simplest)
2. **Fan-out / fan-in** — parallel work that converges into a single reviewer
3. **Selective context** — agents see only the outputs they need, not everything

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All four agents share the same local Ollama model. The differentiation comes entirely from role design — backstory, goal, and task description shape each agent's "personality" and output style.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Crew Orchestration Concept: Role Separation

The key to crew orchestration is **cognitive role separation** — each agent embodies a distinct thinking mode:

| Agent | Thinking Mode | Primary Question | Bias Risk |
|-------|--------------|------------------|-----------|
| PM | **Strategic** — prioritize and scope | "What problem does this solve and for whom?" | Over-scoping, feature creep |
| Marketer | **Creative** — persuade and position | "How do we make people care?" | Hype over substance |
| Analyst | **Analytical** — measure and compare | "What do the numbers say?" | Analysis paralysis |
| QA Reviewer | **Critical** — validate and unify | "Does this all fit together?" | Nitpicking vs. blocking |

**Orchestration insight:** These modes are deliberately in tension. The PM wants to launch fast; the analyst wants more data; the marketer wants bold claims; the QA reviewer challenges everything. This tension produces better output than any single agent could.

### 3.1 Product Manager Agent

**Orchestration role:** The **strategic anchor**. Every downstream agent receives the PM's launch strategy as their foundational context. The PM defines the "north star" that keeps the crew aligned.

**Design rationale:** The backstory emphasizes structured frameworks (positioning canvas, success metrics, RICE scoring) because unstructured strategy produces vague output that downstream agents can't build on.

In [ ]:
pm_agent = Agent(
    role="Senior Product Manager & Launch Strategist",
    goal=(
        "Define the complete launch strategy for the product: target audience, "
        "positioning statement, key value propositions, success metrics (KPIs), "
        "launch timeline with milestones, and risk factors. Produce a structured "
        "launch strategy document that downstream agents can build upon."
    ),
    backstory=(
        "You are a senior product manager with 11 years of experience launching "
        "B2B SaaS and developer tools products. You have led 25+ product launches "
        "ranging from minor feature releases to major platform launches at companies "
        "like Stripe, Datadog, and Figma. Your approach is framework-driven: "
        "(1) Positioning Canvas — category, target buyer, key differentiator, proof "
        "points, (2) Audience Segmentation — primary, secondary, and adjacent audiences "
        "with distinct needs and messaging hooks, (3) Success Metrics — RICE-scored KPIs "
        "with 30/60/90 day targets, (4) Launch Tiers — you classify every launch as Tier 1 "
        "(major — press, events, campaigns), Tier 2 (medium — blog, email, social), or "
        "Tier 3 (minor — changelog, docs). Your launch strategies are specific enough "
        "that a marketer can build a campaign directly from them, and an analyst can "
        "validate the market sizing. You never ship a strategy without explicit "
        "success criteria and a kill/pivot threshold."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {pm_agent.role}")

### 3.2 Marketing Strategist Agent

**Orchestration role:** The **campaign builder**. Transforms the PM's strategic positioning into concrete go-to-market actions — channels, messaging, content, and budget.

**Design rationale:** The backstory specifies channel expertise with expected performance benchmarks. This prevents the common LLM behavior of listing channels without substance ("use social media, email, and content marketing").

In [ ]:
marketer_agent = Agent(
    role="Growth Marketing Strategist & Campaign Architect",
    goal=(
        "Design a complete go-to-market campaign plan based on the PM's launch "
        "strategy. Specify channels, messaging per audience segment, content "
        "calendar, launch sequence, budget allocation, and expected performance "
        "benchmarks for each channel."
    ),
    backstory=(
        "You are a growth marketing strategist with 9 years of experience in B2B "
        "SaaS marketing. You have managed launch campaigns with budgets from $10K to "
        "$500K across Product Hunt, Hacker News, LinkedIn, Twitter/X, email sequences, "
        "webinars, paid search, and developer communities. Your specialty is sequenced "
        "launch campaigns: (1) Pre-launch — build anticipation with waitlists, teasers, "
        "and early access programs, (2) Launch day — coordinated multi-channel blitz with "
        "hour-by-hour timeline, (3) Post-launch — sustained momentum through content, "
        "case studies, and retargeting. For each channel you specify: target audience "
        "segment, messaging angle, content format, expected reach, estimated conversion "
        "rate, and cost. You think in terms of a content cascade: one hero piece (demo "
        "video or blog post) spawns 10+ derivative assets (social clips, email snippets, "
        "community posts). You always include a messaging matrix mapping each audience "
        "segment to their primary pain point, value prop, and proof point."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {marketer_agent.role}")

### 3.3 Market Analyst Agent

**Orchestration role:** The **evidence provider**. Grounds the PM's strategy and marketer's claims in market data — sizing, competitive positioning, and pricing benchmarks.

**Design rationale:** The backstory mandates specific analytical frameworks (TAM/SAM/SOM, competitive matrix, pricing analysis) to ensure the output is structured and actionable rather than a generic market overview.

In [ ]:
analyst_agent = Agent(
    role="Market Research Analyst & Competitive Intelligence Specialist",
    goal=(
        "Provide market intelligence to validate and strengthen the launch strategy: "
        "total addressable market (TAM/SAM/SOM) sizing, competitive landscape mapping, "
        "pricing analysis versus competitors, and identification of market timing "
        "factors that affect launch success."
    ),
    backstory=(
        "You are a market research analyst with 8 years of experience at a top-tier "
        "strategy consulting firm (McKinsey, BCG, or Bain). You specialize in technology "
        "market analysis and competitive intelligence. Your analytical toolkit includes: "
        "(1) Market Sizing — top-down (industry reports) and bottom-up (unit economics) "
        "TAM/SAM/SOM estimation with explicit assumptions, (2) Competitive Matrix — map "
        "competitors on 2x2 grids (e.g., ease-of-use vs. power, price vs. features) to "
        "identify whitespace, (3) Pricing Analysis — compare pricing models (per-seat, "
        "usage-based, freemium) and positioning (premium vs. value vs. penetration), "
        "(4) Market Timing — identify tailwinds (regulatory changes, technology shifts, "
        "buyer behavior trends) and headwinds (market saturation, economic downturn) that "
        "affect launch timing. Your analyses always include explicit assumptions, data "
        "confidence levels (High/Medium/Low), and sensitivity ranges. You cite specific "
        "competitors by name with their pricing, positioning, and estimated market share."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {analyst_agent.role}")

### 3.4 QA Reviewer Agent

**Orchestration role:** The **consistency enforcer**. This agent is unique in the crew because it sees ALL prior outputs and its job is to find problems — contradictions, gaps, unsupported claims, and missing pieces.

**Design rationale:** The backstory explicitly defines the QA checklist categories and the output format (issues table with severity ratings). Without this structure, the QA agent tends to either rubber-stamp everything or raise vague concerns. A structured audit framework produces actionable feedback.

In [ ]:
qa_agent = Agent(
    role="Launch Brief QA Reviewer & Consistency Auditor",
    goal=(
        "Audit the complete launch brief for quality, consistency, and completeness. "
        "Check that the marketing plan aligns with the PM's strategy, the analyst's "
        "data supports the claims made, metrics are measurable, timelines are realistic, "
        "and no critical launch elements are missing. Produce a final audit report with "
        "issues, severity ratings, and a READY / NEEDS REVISION / NOT READY verdict."
    ),
    backstory=(
        "You are a launch operations lead who has QA'd 40+ product launches at a "
        "hypergrowth SaaS company. You've seen launches fail because of misaligned "
        "messaging, unrealistic timelines, missing stakeholder sign-offs, and claims "
        "that couldn't be substantiated. Your QA process evaluates launch briefs across "
        "7 dimensions: (1) Strategic Alignment — does the GTM plan actually serve the "
        "stated positioning and audience? (2) Consistency — do all sections tell the same "
        "story? (audience, pricing, messaging), (3) Completeness — are all required launch "
        "brief sections present? (4) Evidence — are market claims backed by analyst data? "
        "(5) Feasibility — are timelines, budgets, and resource assumptions realistic? "
        "(6) Metrics — are success criteria specific, measurable, and time-bound? "
        "(7) Risk Coverage — are launch risks identified with contingency plans? For each "
        "issue found, you assign a severity: Critical (blocks launch), Major (degrades "
        "launch quality), or Minor (nice to fix). You always provide a final verdict: "
        "READY (ship it), NEEDS REVISION (fix these specific issues), or NOT READY "
        "(fundamental problems exist)."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {qa_agent.role}")

### Agent Roster Summary

In [ ]:
agents = [pm_agent, marketer_agent, analyst_agent, qa_agent]

print("=" * 70)
print("PRODUCT LAUNCH CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)

## 4. Define Tasks & Orchestration

### Crew Orchestration Deep Dive

Orchestration answers three questions:
1. **What order?** — Which agent goes first, second, third?
2. **What context?** — Which agents see which prior outputs?
3. **What convergence?** — How do separate outputs merge into one deliverable?

#### Orchestration Pattern: Plan → Build → Verify

| Phase | Agent(s) | Purpose | Context Received |
|-------|---------|---------|-----------------|
| **PLAN** | PM | Set strategy and scope | Product brief only |
| **BUILD** | Marketer + Analyst | Create campaign + validate with data | PM's strategy |
| **VERIFY** | QA Reviewer | Audit everything for consistency | All three prior outputs |

**Why this pattern works for launch briefs:**
- The PM creates the **single source of truth** for positioning and audience
- The Marketer and Analyst both build from the PM's strategy, ensuring alignment
- The QA Reviewer catches any drift or contradiction that crept in during the build phase

**Key orchestration decision:** In this first run, we use a linear pipeline (PM → Marketer → Analyst → QA). In Section 10, we'll show the fan-out version where Marketer and Analyst work in parallel with explicit context.

### 4.1 Configure the Product

In [ ]:
# =====================================================
# CONFIGURE YOUR PRODUCT HERE
# =====================================================
PRODUCT = {
    "name": "FlowForge",
    "tagline": "AI-Powered Workflow Automation for DevOps Teams",
    "category": "Developer Tools / DevOps",
    "description": (
        "FlowForge is a no-code/low-code platform that lets DevOps teams build, "
        "test, and deploy CI/CD pipelines using natural language. Instead of writing "
        "YAML configs, teams describe their desired workflow in plain English and "
        "FlowForge generates, validates, and maintains the pipeline configuration."
    ),
    "target_market": "Mid-market B2B SaaS companies (100-1000 employees) with DevOps teams",
    "pricing": "Freemium — free for 3 pipelines, $49/user/mo for Pro, Enterprise custom",
    "stage": "Public launch (previously in 6-month private beta with 120 teams)",
    "key_features": [
        "Natural language pipeline builder",
        "Automatic YAML generation and validation",
        "Multi-cloud support (AWS, GCP, Azure)",
        "Built-in testing and rollback",
        "Compliance templates (SOC2, HIPAA)",
    ],
    "launch_date": "6 weeks from today",
}
# =====================================================

print("PRODUCT BRIEF")
print("=" * 60)
for k, v in PRODUCT.items():
    label = k.replace("_", " ").title()
    if isinstance(v, list):
        print(f"  {label}:")
        for item in v:
            print(f"    • {item}")
    else:
        val = str(v)
        if len(val) > 70:
            print(f"  {label}:")
            print(f"    {val}")
        else:
            print(f"  {label:<20} {val}")

### 4.2 Task 1 — Launch Strategy (PM)

**Orchestration phase:** PLAN — the PM defines the strategic foundation that all other agents build on.

**Context:** Only the raw product brief. The PM works from first principles.

In [ ]:
product_brief = (
    f"Product: {PRODUCT['name']} — {PRODUCT['tagline']}\n"
    f"Category: {PRODUCT['category']}\n"
    f"Description: {PRODUCT['description']}\n"
    f"Target Market: {PRODUCT['target_market']}\n"
    f"Pricing: {PRODUCT['pricing']}\n"
    f"Stage: {PRODUCT['stage']}\n"
    f"Key Features: {', '.join(PRODUCT['key_features'])}\n"
    f"Launch Date: {PRODUCT['launch_date']}"
)

task_strategy = Task(
    description=(
        f"Define the complete launch strategy for this product:\n\n"
        f"{product_brief}\n\n"
        "Your launch strategy MUST include:\n"
        "1. **Launch Tier Classification**: Tier 1 (major) / Tier 2 (medium) / Tier 3 (minor) "
        "   — with justification\n"
        "2. **Positioning Statement**: For [target], [product] is a [category] that [key benefit] "
        "   unlike [alternative] because [differentiator]\n"
        "3. **Audience Segmentation**: Primary, secondary, and adjacent audiences with distinct "
        "   needs and messaging hooks for each\n"
        "4. **Value Propositions**: Top 3 value props ranked by audience resonance, each with "
        "   a proof point from beta data\n"
        "5. **Success Metrics**: 5-7 KPIs with 30/60/90 day targets (e.g., signups, activations, "
        "   paid conversions, NPS, press mentions)\n"
        "6. **Launch Timeline**: Week-by-week milestones for the 6-week launch window\n"
        "7. **Risk Factors**: Top 3 launch risks with probability, impact, and mitigation plans\n"
        "8. **Kill/Pivot Threshold**: What specific metrics would trigger a pivot or wind-down?\n"
    ),
    expected_output=(
        "A structured launch strategy document with launch tier, positioning statement, "
        "audience segments, value propositions, KPIs with targets, week-by-week timeline, "
        "risk factors, and kill/pivot thresholds."
    ),
    agent=pm_agent,
)

print(f"Task created: Launch Strategy -> {task_strategy.agent.role}")

### 4.3 Task 2 — Go-to-Market Campaign (Marketer)

**Orchestration phase:** BUILD — transforms strategy into actionable marketing plan.

**Context:** Receives the PM's launch strategy. Must align channel selection, messaging, and timing with the PM's audience segments and timeline.

In [ ]:
task_campaign = Task(
    description=(
        "Design a complete go-to-market campaign plan based on the PM's launch "
        "strategy for:\n\n"
        f"{product_brief}\n\n"
        "Your campaign plan MUST include:\n"
        "1. **Messaging Matrix**: For each audience segment identified by the PM, specify:\n"
        "   - Primary pain point\n"
        "   - Value proposition (must match PM's value props)\n"
        "   - Proof point / social proof\n"
        "   - Call-to-action\n"
        "2. **Channel Strategy**: For each channel, specify:\n"
        "   - Target audience segment\n"
        "   - Content format (blog post, video, social thread, email, etc.)\n"
        "   - Expected reach and conversion estimate\n"
        "   - Budget allocation\n"
        "3. **Content Cascade**: Define the hero content piece and 8-10 derivative assets\n"
        "4. **Launch Sequence**: Hour-by-hour launch day plan + pre-launch and post-launch phases\n"
        "5. **Budget Breakdown**: Total budget with allocation by channel (suggest $30-50K range)\n"
        "6. **Community Strategy**: Plan for Product Hunt, Hacker News, Reddit, Dev.to, "
        "   and relevant Discord/Slack communities\n"
    ),
    expected_output=(
        "A go-to-market campaign plan with messaging matrix, channel strategy with "
        "budgets, content cascade, launch day sequence, budget breakdown, and "
        "community strategy."
    ),
    agent=marketer_agent,
)

print(f"Task created: GTM Campaign -> {task_campaign.agent.role}")

### 4.4 Task 3 — Market Intelligence (Analyst)

**Orchestration phase:** BUILD — validates the strategy with market data and competitive intelligence.

**Context:** Receives both the PM's strategy and the marketer's campaign plan, allowing the analyst to reality-check claims made in both.

In [ ]:
task_analysis = Task(
    description=(
        "Provide market intelligence to validate the launch strategy and campaign "
        "plan for:\n\n"
        f"{product_brief}\n\n"
        "Your market analysis MUST include:\n"
        "1. **Market Sizing** (TAM/SAM/SOM):\n"
        "   - Total Addressable Market with methodology and assumptions\n"
        "   - Serviceable Addressable Market (realistic)\n"
        "   - Serviceable Obtainable Market (12-month target)\n"
        "2. **Competitive Landscape**:\n"
        "   - 4-6 direct and adjacent competitors with positioning, pricing, and strengths\n"
        "   - 2x2 competitive matrix (suggest: ease-of-use vs. power)\n"
        "   - White-space opportunities the product can own\n"
        "3. **Pricing Validation**:\n"
        "   - Compare the freemium + $49/user/mo model against competitor pricing\n"
        "   - Is the pricing too high, too low, or positioned correctly?\n"
        "   - Freemium conversion rate benchmark for this category\n"
        "4. **Market Timing Assessment**:\n"
        "   - Tailwinds favoring this launch\n"
        "   - Headwinds that could slow adoption\n"
        "   - Timing grade: Excellent / Good / Neutral / Poor\n"
        "5. **Claim Validation**: Are the value propositions and projected metrics from "
        "   the PM's strategy realistic given market conditions?\n"
    ),
    expected_output=(
        "A market intelligence report with TAM/SAM/SOM sizing, competitive matrix "
        "with 4-6 competitors, pricing validation, market timing assessment, and "
        "claim validation against the PM's strategy."
    ),
    agent=analyst_agent,
)

print(f"Task created: Market Intelligence -> {task_analysis.agent.role}")

### 4.5 Task 4 — QA Audit (QA Reviewer)

**Orchestration phase:** VERIFY — the QA reviewer sees ALL three prior outputs and audits the complete launch brief for quality, consistency, and completeness.

**Why a dedicated QA agent?** Without an explicit verification step, multi-agent systems accumulate small inconsistencies. The marketer might target an audience the PM didn't define. The analyst might size a different market than the PM positioned for. The QA agent catches these drift errors.

In [ ]:
task_qa = Task(
    description=(
        "Audit the complete launch brief for quality, consistency, and launch readiness.\n\n"
        f"Product: {PRODUCT['name']} — {PRODUCT['tagline']}\n\n"
        "You have received outputs from three agents:\n"
        "1. PM's Launch Strategy (positioning, audience, metrics, timeline)\n"
        "2. Marketer's GTM Campaign (channels, messaging, budget, content)\n"
        "3. Analyst's Market Intelligence (TAM/SAM/SOM, competitors, pricing)\n\n"
        "Audit across 7 dimensions:\n"
        "1. **Strategic Alignment**: Does the GTM plan serve the PM's stated positioning?\n"
        "2. **Consistency**: Do all sections define the same audience, pricing, and messaging?\n"
        "3. **Completeness**: Are all required launch brief sections present?\n"
        "4. **Evidence**: Are market claims backed by the analyst's data?\n"
        "5. **Feasibility**: Are timelines and budgets realistic for the team size?\n"
        "6. **Metrics**: Are success criteria specific, measurable, and time-bound?\n"
        "7. **Risk Coverage**: Are launch risks identified with contingency plans?\n\n"
        "For EACH issue found, provide:\n"
        "- Issue description\n"
        "- Severity: Critical / Major / Minor\n"
        "- Which section(s) are affected\n"
        "- Recommended fix\n\n"
        "Conclude with:\n"
        "- Issue summary count (Critical / Major / Minor)\n"
        "- Overall verdict: READY / NEEDS REVISION / NOT READY\n"
        "- Top 3 actions required before launch\n"
    ),
    expected_output=(
        "A QA audit report with issues rated by severity across 7 dimensions, "
        "an issue summary count, a READY/NEEDS REVISION/NOT READY verdict, and "
        "top 3 required actions."
    ),
    agent=qa_agent,
)

print(f"Task created: QA Audit -> {task_qa.agent.role}")

### Orchestration Flow Visualization

In [ ]:
tasks = [task_strategy, task_campaign, task_analysis, task_qa]
task_names = ["Launch Strategy", "GTM Campaign", "Market Intelligence", "QA Audit"]
phases = ["PLAN", "BUILD", "BUILD", "VERIFY"]
flow_labels = [
    "positioning, audience, KPIs, timeline",
    "channels, messaging, content, budget",
    "TAM/SAM, competitors, pricing validation",
    "consistency audit, gap check, verdict",
]

print("LAUNCH BRIEF ORCHESTRATION FLOW")
print("=" * 70)
for i, (name, task, phase, flow) in enumerate(zip(task_names, tasks, phases, flow_labels)):
    print(f"  [{phase}] {name}")
    print(f"    Agent: {task.agent.role}")
    print(f"    Output: {flow}")
    if i < len(tasks) - 1:
        connector = "│  (context accumulates ↓)" if i < 2 else "│  (ALL outputs converge ↓)"
        print(f"    {'':8}{connector}")
print("=" * 70)
print(f"\nQA Reviewer sees outputs from ALL 3 prior agents")

## 5. Assemble and Run the Crew

### Orchestration Mode: Sequential Pipeline

In sequential mode, CrewAI runs tasks in order and accumulates context:

```
Task 1 output  ────────────────────────────────────> Task 2 context
Task 1 + 2 outputs  ──────────────────────────────> Task 3 context
Task 1 + 2 + 3 outputs  ─────────────────────────> Task 4 context
```

**Trade-off:** Sequential is simple and ensures consistency (everyone builds on the same foundation), but it's slower than parallel execution and later agents may be biased by earlier ones' framing.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
# Execute the launch brief pipeline
print("=" * 70)
print(f"LAUNCHING PRODUCT LAUNCH CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Product: {PRODUCT['name']} — {PRODUCT['tagline']}")
print(f"Orchestration: PM -> Marketer -> Analyst -> QA (sequential)")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — QA Audit & Verdict

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — QA Audit & Launch Readiness Verdict")
print("=" * 70)
print(result.raw)

### 6.2 Individual Phase Outputs

Trace how the launch brief was built — from strategy through campaign design, market validation, and quality audit.

In [ ]:
for phase, name, task in zip(phases, task_names, tasks):
    print("\n" + "=" * 70)
    print(f"[{phase}] {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("LAUNCH BRIEF PIPELINE METRICS")
print("=" * 65)
print(f"{'Phase':<8} {'Task':<22} {'Agent':<28} {'Output':>8}")
print("-" * 65)

total = 0
for phase, name, task in zip(phases, task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    agent_short = task.agent.role.split("&")[0].strip()[:26]
    print(f"{phase:<8} {name:<22} {agent_short:<28} {length:>6,}")

print("-" * 65)
print(f"{'':8} {'TOTAL':<50} {total:>6,}")
print(f"\nProduct: {PRODUCT['name']} — {PRODUCT['tagline']}")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"Tokens: {result.token_usage.get('total_tokens', 'N/A')}")

## 7. Save Launch Brief

In [ ]:
# Build and save the unified launch brief
brief_lines = [
    f"# Launch Brief: {PRODUCT['name']}",
    f"**{PRODUCT['tagline']}**",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Launch Date:** {PRODUCT['launch_date']}",
    "---",
]

section_titles = [
    "Launch Strategy (PM)",
    "Go-to-Market Campaign (Marketing)",
    "Market Intelligence (Analyst)",
    "QA Audit & Launch Readiness (QA)",
]
for title, task in zip(section_titles, tasks):
    brief_lines.append(f"\n## {title}\n")
    brief_lines.append(task.output.raw if task.output else "[No output]")
    brief_lines.append("\n---")

brief = "\n".join(brief_lines)
with open("launch_brief.md", "w", encoding="utf-8") as f:
    f.write(brief)

print(f"Launch brief saved: launch_brief.md ({len(brief):,} chars)")

## 8. Experiment: Different Product

Re-run the same crew on a completely different product to demonstrate pipeline reusability.

In [ ]:
# =====================================================
# SECOND PRODUCT
# =====================================================
PRODUCT_2 = {
    "name": "HealthPulse",
    "tagline": "AI Sleep Coach for Remote Teams",
    "category": "B2B Wellness / HR Tech",
    "description": (
        "HealthPulse is a Slack/Teams app that uses wearable data (Fitbit, Apple Watch, "
        "Oura) to provide personalized sleep coaching to remote team members. It gives "
        "managers anonymized team wellness dashboards without exposing individual data."
    ),
    "target_market": "Remote-first B2B companies (50-500 employees) with wellness budgets",
    "pricing": "Free pilot (30 days), $8/user/mo team plan, $15/user/mo with manager dashboards",
    "stage": "Beta launch (60 beta users across 5 companies, 4.2/5 satisfaction)",
    "key_features": [
        "Wearable data sync (Fitbit, Apple Watch, Oura Ring)",
        "Personalized sleep improvement nudges via Slack/Teams",
        "Team wellness dashboard (anonymized/aggregated)",
        "Weekly sleep score trends and recommendations",
        "Integration with HR wellness program reporting",
    ],
    "launch_date": "4 weeks from today",
}

print("PRODUCT 2 BRIEF")
print("=" * 60)
for k, v in PRODUCT_2.items():
    label = k.replace("_", " ").title()
    if isinstance(v, list):
        print(f"  {label}:")
        for item in v:
            print(f"    • {item}")
    else:
        val = str(v)
        if len(val) > 70:
            print(f"  {label}:")
            print(f"    {val}")
        else:
            print(f"  {label:<20} {val}")

In [ ]:
brief2 = (
    f"Product: {PRODUCT_2['name']} — {PRODUCT_2['tagline']}\n"
    f"Category: {PRODUCT_2['category']}\n"
    f"Description: {PRODUCT_2['description']}\n"
    f"Target Market: {PRODUCT_2['target_market']}\n"
    f"Pricing: {PRODUCT_2['pricing']}\n"
    f"Stage: {PRODUCT_2['stage']}\n"
    f"Key Features: {', '.join(PRODUCT_2['key_features'])}\n"
    f"Launch Date: {PRODUCT_2['launch_date']}"
)

task2_strategy = Task(
    description=(
        f"Define launch strategy for:\n{brief2}\n\n"
        "Include: launch tier, positioning statement, audience segments, value props, "
        "KPIs with 30/60/90 day targets, timeline, risk factors, and kill/pivot thresholds."
    ),
    expected_output="Launch strategy with positioning, audience, KPIs, and timeline.",
    agent=pm_agent,
)

task2_campaign = Task(
    description=(
        f"Design GTM campaign for:\n{brief2}\n\n"
        "Include: messaging matrix per segment, channel strategy with budget, "
        "content cascade, launch day sequence, and community strategy. "
        "Note: this is a wellness/HR product — channel mix differs from dev tools."
    ),
    expected_output="GTM campaign with channels, messaging, budget, and launch sequence.",
    agent=marketer_agent,
)

task2_analysis = Task(
    description=(
        f"Market intelligence for:\n{brief2}\n\n"
        "Include: TAM/SAM/SOM for HR wellness tech, competitive landscape (Calm, "
        "Headspace for Work, Virgin Pulse, etc.), pricing validation, and market timing."
    ),
    expected_output="Market analysis with sizing, competitors, pricing validation, timing.",
    agent=analyst_agent,
)

task2_qa = Task(
    description=(
        f"QA audit for the launch brief of: {PRODUCT_2['name']}\n\n"
        "Audit all prior outputs for consistency, completeness, evidence backing, "
        "feasibility, and measurable metrics. Pay special attention to privacy "
        "concerns (wearable health data) that must be addressed in the brief. "
        "Deliver READY / NEEDS REVISION / NOT READY verdict."
    ),
    expected_output="QA audit with severity-rated issues and launch readiness verdict.",
    agent=qa_agent,
)

tasks_2 = [task2_strategy, task2_campaign, task2_analysis, task2_qa]
print(f"Product 2 tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Product: {PRODUCT_2['name']} — {PRODUCT_2['tagline']}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"QA VERDICT — {PRODUCT_2['name']}")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Launch Briefs

In [ ]:
print("LAUNCH BRIEF COMPARISON")
print("=" * 72)
print(f"{'Phase':<8} {'Task':<22} {'FlowForge':>15} {'HealthPulse':>15}")
print("-" * 72)

for phase, name, t1, t2 in zip(phases, task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{phase:<8} {name:<22} {len1:>12,} ch {len2:>12,} ch")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 72)
print(f"{'':8} {'TOTAL':<22} {total1:>12,} ch {total2:>12,} ch")

print(f"\nProduct 1: {PRODUCT['name']} — {PRODUCT['category']}")
print(f"Product 2: {PRODUCT_2['name']} — {PRODUCT_2['category']}")

## 10. Advanced: Fan-Out Orchestration with Explicit Context

### Problem with Pure Sequential

In the sequential pipeline, the Analyst sees the Marketer's output. This can create **anchoring bias** — the Analyst's market sizing might be unconsciously influenced by the Marketer's claims rather than derived independently.

### Solution: Fan-Out / Fan-In Pattern

```
                        ┌──> Marketer  ──┐
PM (strategy)  ─────────┤                ├──> QA Reviewer
                        └──> Analyst   ──┘
```

The Marketer and Analyst both build from the PM's strategy but do NOT see each other's work. The QA Reviewer then reconciles both independent outputs, catching contradictions that sequential execution might hide.

**This is a key orchestration pattern:** Use fan-out when you want **independent perspectives** on the same input, and fan-in when you need a **single reviewer** to reconcile them.

In [ ]:
# Fan-out orchestration with explicit context control

fan_strategy = Task(
    description=(
        f"Define launch strategy for: {PRODUCT['name']}\n{product_brief}\n\n"
        "Include: launch tier, positioning, audience segments, value props, "
        "KPIs with targets, timeline, and risk factors."
    ),
    expected_output="Launch strategy document.",
    agent=pm_agent,
)

# Marketer sees ONLY the PM's strategy
fan_campaign = Task(
    description=(
        f"Design GTM campaign for: {PRODUCT['name']}\n{product_brief}\n\n"
        "Include: messaging matrix, channel strategy, content cascade, "
        "launch sequence, and budget breakdown."
    ),
    expected_output="GTM campaign plan.",
    agent=marketer_agent,
    context=[fan_strategy],  # Only sees PM strategy
)

# Analyst sees ONLY the PM's strategy (NOT the marketer's campaign)
fan_analysis = Task(
    description=(
        f"Market intelligence for: {PRODUCT['name']}\n{product_brief}\n\n"
        "Include: TAM/SAM/SOM, competitive matrix, pricing validation, "
        "and market timing. Work INDEPENDENTLY — do not reference the "
        "marketing campaign."
    ),
    expected_output="Independent market analysis.",
    agent=analyst_agent,
    context=[fan_strategy],  # Only sees PM strategy (NOT marketer)
)

# QA sees ALL three — reconciles independent assessments
fan_qa = Task(
    description=(
        f"QA audit for: {PRODUCT['name']} launch brief\n\n"
        "IMPORTANT: The Marketer and Analyst worked INDEPENDENTLY from the PM's "
        "strategy. They did NOT see each other's outputs. Your job is to:\n"
        "1. Check if the Marketer's campaign aligns with the Analyst's market data\n"
        "2. Identify contradictions between their independent assessments\n"
        "3. Flag where the Marketer made claims the Analyst's data doesn't support\n"
        "4. Deliver a READY / NEEDS REVISION / NOT READY verdict\n"
    ),
    expected_output="Reconciled QA audit with cross-check of independent outputs.",
    agent=qa_agent,
    context=[fan_strategy, fan_campaign, fan_analysis],  # Sees everything
)

fan_tasks = [fan_strategy, fan_campaign, fan_analysis, fan_qa]

print("FAN-OUT ORCHESTRATION FLOW")
print("=" * 60)
print("                   ┌──> Marketer (sees PM only)  ──┐")
print("  PM (strategy) ───┤                               ├──> QA (reconciles)")
print("                   └──> Analyst (sees PM only)   ──┘")
print()
fan_names = ["PM Strategy", "GTM Campaign (independent)", "Market Intel (independent)", "QA Audit (reconciled)"]
for i, (name, task) in enumerate(zip(fan_names, fan_tasks), 1):
    ctx = [t.agent.role.split("&")[0].strip()[:25] for t in (task.context or [])]
    ctx_str = " + ".join(ctx) if ctx else "none (product brief only)"
    print(f"  Task {i}: {name}")
    print(f"    Context from: {ctx_str}")

In [ ]:
# Run the fan-out crew
crew_fanout = Crew(
    agents=agents,
    tasks=fan_tasks,
    process=Process.sequential,  # Sequential execution, but context controls data flow
    verbose=True,
    memory=False,
)

print(f"Launching fan-out crew — {datetime.now().strftime('%H:%M:%S')}")
result_fanout = crew_fanout.kickoff()
print(f"\nFan-out crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Compare fan-out results
print("FAN-OUT ORCHESTRATION RESULTS")
print("=" * 65)
for name, task in zip(fan_names, fan_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = [t.agent.role.split("&")[0].strip()[:20] for t in (task.context or [])]
    ctx_str = ", ".join(ctx) if ctx else "auto"
    print(f"  {name:<35} {length:>6,} chars | Context: {ctx_str}")

## 11. Crew Orchestration Patterns — Reference Guide

### Pattern 1: Sequential Pipeline (Sections 5-6)

```
A → B → C → D
```

Each agent sees ALL prior outputs. Simple, predictable, but creates anchoring bias.

| Pros | Cons |
|------|------|
| Easy to implement | Later agents anchored by earlier framing |
| Natural context accumulation | Bottleneck — one agent at a time |
| Good for dependent tasks | Error propagation — bad Task 1 → bad everything |

**Best for:** Workflows where each step genuinely depends on the previous one.

### Pattern 2: Fan-Out / Fan-In (Section 10)

```
     ┌──> B ──┐
A ───┤        ├──> D
     └──> C ──┘
```

B and C work independently from A's output. D reconciles.

| Pros | Cons |
|------|------|
| Independent perspectives | More complex context management |
| Reduces anchoring bias | D must handle contradictions |
| Parallel-ready | Need explicit `context` parameter |

**Best for:** When you want independent analysis from multiple angles.

### Pattern 3: Iterative Refinement

```
A → B → C → B (revised) → D
```

The QA agent sends work back for revision. CrewAI supports this with callback mechanisms.

| Pros | Cons |
|------|------|
| Self-correcting | More complex, more LLM calls |
| Converges to quality | Risk of infinite loops |
| Mimics real review cycles | Need clear termination criteria |

**Best for:** High-stakes documents that need multiple drafts.

### Pattern 4: Specialist Consultation

```
A → B → [B asks A a follow-up] → B continues → C
```

Agents can delegate sub-questions to other agents mid-task. Enabled via `allow_delegation=True`.

| Pros | Cons |
|------|------|
| Natural collaboration | Unpredictable execution flow |
| Agents fill their own gaps | Hard to debug |
| Richer outputs | Token-expensive |

**Best for:** Exploratory tasks where agents need to ask clarifying questions.

### Choosing the Right Pattern

| Scenario | Recommended Pattern |
|----------|-------------------|
| Linear workflow with clear dependencies | Sequential |
| Multiple independent analyses of the same input | Fan-out / Fan-in |
| Quality-critical documents (legal, compliance) | Iterative Refinement |
| Open-ended research or brainstorming | Specialist Consultation |
| Launch briefs, reports, proposals | Fan-out with QA reconciliation |

## 12. Key Takeaways

### What We Built
- A **4-agent product launch crew** (PM → Marketer → Analyst → QA) producing a unified launch brief
- Ran it on **two products** (DevOps tool + HR wellness app) across different B2B verticals
- Demonstrated **fan-out orchestration** where Marketer and Analyst work independently with QA reconciliation

### Crew Orchestration Concepts
1. **Plan → Build → Verify** — Strategy first (PM), then parallel execution (Marketer + Analyst), then quality check (QA)
2. **Context control is orchestration** — The `context` parameter determines which agents see which outputs, preventing bias
3. **Fan-out reduces anchoring** — Independent agents produce diverse perspectives; a reconciler catches contradictions
4. **QA as a structural role** — Without explicit verification, multi-agent systems accumulate drift between sections

### Agent Design Lessons
- **PM Agent:** Define structured frameworks (positioning canvas, RICE scoring) so downstream agents get actionable input
- **Marketer Agent:** Require a messaging matrix mapping audiences to pain points — prevents generic "use social media" output
- **Analyst Agent:** Mandate explicit assumptions and confidence levels — LLMs otherwise present estimates as facts
- **QA Agent:** Define the audit checklist in the backstory — without structure, QA output oscillates between rubber-stamping and vague criticism

### Production Tips
- Use `output_pydantic` with a `LaunchBrief` schema to enforce structured JSON output across all agents
- Enable `memory=True` to retain lessons across multiple product launches
- Add a **PR/Communications Agent** for press release and media outreach planning
- Consider an **Engineering Liaison Agent** that validates feature claims against the actual product capabilities
- For real launches, connect the Analyst to market data APIs (Crunchbase, G2, SimilarWeb) for ground truth